---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-14: Multi-Modality of AI Models (Part-II)</h1>

# Learning agenda of this notebook
1. Speech Recognition: Audio → Text
    - OpenAI's Whisper
    - Using OpenAI's whisper-base with Transformers Pipeline
2. Speech Generation: Text → Speech
    - Using OpenAI's tts-1
    - Using Google Text-to-Speech (gTTS)
    - Using Open-Source Models with Transformers Pipeline
3. Project (Cross-Modal Chaining)
    - Speak: “Describe this picture of a cat.”
    - Model transcribes voice (Whisper).
    - Model analyzes the image (GPT-4o).
    - Model generates a narration in text.
    - Model speaks it back (TTS).
6. To Do:
    - Video → Text (Video Understanding)
    - Text → Video Generation

In [2]:
#  Load the API  API Keys
import os
from dotenv import load_dotenv
load_dotenv('../keys/.env', override=True) 

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv('HF_TOKEN')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:8]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

if hf_token:
    print(f"Hugging Face Tokens exists and begins {hf_token[:8]}")
else:
    print("Hugging Face tokens not set")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-a
Google API Key exists and begins AIzaSyDA
Groq API Key exists and begins gsk_LyFp
Hugging Face Tokens exists and begins hf_oEyHP


# 1. <span style='background :lightgreen' >Speech Recognition: Audio → Text</span>

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Audio-to-text models use automatic speech recognition (ASR) to transcribe spoken language into written text, supporting real-time transcription, multilingual speech, and speaker-specific insights for accessibility, productivity, and audio analysis.</div></h2>

### Closed Source:
- **OpenAI Whisper API:** Hosted version of Whisper offering accurate multilingual transcription and translation through OpenAI’s API. (https://openai.com/research/whisper)
- **Google Speech-to-Text:** Cloud service offering real-time transcription, speaker diarization, and support for over 125 languages. (https://cloud.google.com/speech-to-text)
- **Azure Speech Services:** Microsoft’s enterprise-ready speech recognition with customizable models and integration into Azure AI. (https://azure.microsoft.com/en-us/products/ai-services/speech-to-text)
### Open Source:
- **Whisper (local):** Open-source ASR (Automatic Speech Recognition) model by OpenAI supporting multilingual transcription and translation. (https://github.com/openai/whisper)
- **Wav2Vec2:** Facebook AIs self-supervised speech representation model, fine-tuned for high-quality ASR (Automatic Speech Recognition). (https://huggingface.co/facebook/wav2vec2-base-960h)
- **SpeechT5:** Microsoft’s transformer-based model handling ASR (Automatic Speech Recognition), TTS (Text-to-Speech), and speech translation. (https://huggingface.co/microsoft/speecht5_asr)

## a.  OpenAI's Whisper:
- The **OpenAI's Whisper** model is most commonly used for transcribing audio recordings, generating subtitles for videos, converting spoken content into written documents.
- This code uses OpenAI’s hosted whisper-1 model (a closed-source, cloud-run version of Whisper) to transcribe a local audio file by sending it to OpenAI’s API.
- OpenAI client provides two endpoints for the Whisper model:
    - **`audio.transcriptions.create()`:** Converts speech → text in the same language. Performs automatic speech recognition (ASR). Example: English audio → English text, Urdu audio → Urdu text and so on
    - **`audio.translations.create()`:** Converts speech (in any language) → English text. Example: Urdu audio → English text (Cannot output in language other than English as of today)
- It detects the language automatically and produces text in the language spoken in the audio. Has support of 90+ languages like English (en), Urdu (ur), Bengali (bn), Arabic (ar), French (fr), German (de), Japanese (ja), Hindi (hi), Hebrew (he) and so on
- Practically you can use this model to transcribe audio into whatever language the audio is in or to translate and transcribe the audio into English.

### Example 1 (Generating Transcriptions with `whisper-1`)

In [2]:
# Play an English audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-english-audio.mp3', autoplay=False)

In [4]:
# Converts English speech to English Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.

audio_file = open('../data/audios/arif-english-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language=None                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'Assalam-o-Alaikum Students, I welcome you all to Learning AI with Arif Butt.\n'

In [4]:
# Play an Urdu audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [8]:
# Converts Urdu speech to Urdu Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.


audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language=None                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'عزیز طلبہ السلام علیکم میں آپ سب کو جنریٹیو آرٹیفیشل انٹیلیجنز کے کورس میں خوشحامدید کہتا ہوں\n'

In [4]:
# Converts Urdu speech to Urdu Text (same audio file as above, shown with an explicit base_url)
from openai import OpenAI

client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.

audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language="hi"                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'अजीज तलबा अस्सलाम ओनिकम, मैं आप सबको जेनेरेटिव आर्टिफिशल इंटेलिजन्स के कोर्स में खुशामदीत कहता हूं।\n'

### Example 2 (Doing Translations with `whisper-1`)
- OpenAI’s Whisper can take audio in many languages and **translate** it into English text.
- It detects the spoken language automatically and produces an English transcription of the audio.

In [6]:
# Play an Urdu audio file, which we want to translate/transcribe in English
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [6]:
# Translates Urdu speech to English Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
result = client.audio.translations.create(
        model="whisper-1",
        file=audio_file
    )
audio_file.close()
print(result.text) 

Azeez Talba, Assalam-o-Alaikum. I welcome you all to the course on Generative Artificial Intelligence.


In [8]:
# Play a German language audio file, whose English transcription we want to create
from IPython.display import Audio
Audio('../data/audios/german-audio.mp3', autoplay=False)

In [9]:
# Translates German speech to English Text
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=openai_api_key)

audio_file = open('../data/audios/german-audio.mp3', 'rb')
result = client.audio.translations.create(
        model="whisper-1",
        file=audio_file
    )
audio_file.close()
print(result.text) 

Hello? Great, dad! It's really great here with the lions. Only this lion here, your BQ, is not here. He's probably sleeping inside, right? Not today. Because today the lion is not inside, but... ...not even here. But? With the doctor. Once a year he is examined, vaccinated, something like that. I understand. But now tell me, how was it at school? Good. The patch on your forehead, where did it come from? At the school yard. Doesn't matter! Really, dad, doesn't matter! No, it doesn't matter. They said things about you.


## b.  OpenAI's `whisper-base` (Open Source):
- The **`openai/whisper-base`** is an open source multilingual speech recognition model that can be run locally without requiring API calls. It's commonly used for transcribing audio recordings, generating subtitles for videos, and converting spoken content into written documents.
- The model comes in different sizes: tiny, base, small, medium, and large, with varying accuracy and computational requirements. The whisper-base model offers a good balance between speed and accuracy for most use cases.
- The following code when run for the first time will download `openai/whisper-base` model from Hugging Face. The model files are downloaded once and cached for future use.
- The Transformers pipeline provides flexible configuration through the generate_kwargs parameter:
    - task="transcribe": Converts speech → text in the same language. Performs automatic speech recognition (ASR). Example: English audio → English text, Urdu audio → Urdu text and so on
    - task="translate": Converts speech (in any language) → English text only. Example: Urdu audio → English text (Cannot output in language other than English)
- It can automatically detect the language or you can explicitly specify it using language parameter. Supports 90+ languages including English (en), Urdu (ur), Bengali (bn), Arabic (ar), French (fr), German (de), Japanese (ja), Hindi (hi), Hebrew (he) and so on
- HuggingFace: https://huggingface.co/openai/whisper-base
- GitHub Repo: https://github.com/openai/whisper

### Example 1 (Generating Transcriptions with `whisper-base`)

In [2]:
# Play an English audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-english-audio.mp3', autoplay=True)

In [3]:
# Converts English speech to English Text (You may have to install ffmpeg binary on your OS for this code to work as ffmpeg is required to load audio files from filename
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline(task="automatic-speech-recognition", model="openai/whisper-base", device=-1)

# Transcribe the speech to text. The pipeline can accept either a file path or audio data directly
transcript = pipe('../data/audios/arif-english-audio.mp3',
                    generate_kwargs={
                                    "language": "english",  # Specify language of the input audio (can use "urdu", "hindi", etc.)
                                    "task": "transcribe",  # Options: "transcribe" or "translate"
                                    "temperature": 0.0,  # Lower = more deterministic output
                                    }
                    )
# Display the results
print(f"Transcribed Text: {transcript['text']}")

Device set to use cpu


Transcribed Text:  Assalamu alaikum students, I welcome you all to learning AI with RFBUT.


In [2]:
# Play an Urdu audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=True)

In [2]:
# Converts Urdu speech to Urdu Text
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline("automatic-speech-recognition", "openai/whisper-base", device=-1)

# Transcribe the speech to text. The pipeline can accept either a file path or audio data directly
transcript = pipe('../data/audios/arif-urdu-audio.mp3',
    generate_kwargs={
        "language": "urdu",  # Specify language of the input audio (can use "urdu", "hindi", etc.)
        "task": "transcribe",  # Options: "transcribe" or "translate"
        "temperature": 0.0,  # Lower = more deterministic output
    }
)

# Display the results
print(f"Transcribed Text: {transcript['text']}")

Device set to use cpu


Transcribed Text:  ازیز طلبہ اصلام علیکم میں آپ سک کو جنریٹیف ارٹیفشل انٹیلیجنس کے کورس میں خوشام دیت کہتا ہوں


### Example 2 (Doing Translations with `whisper-base`)

In [4]:
# Play an Urdu audio file, whose English translation we want to generate
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [5]:
# Translates Urdu speech to English Text
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline("automatic-speech-recognition", "openai/whisper-base", device=-1)

# Translate the speech to English text
result = pipe('../data/audios/arif-urdu-audio.mp3',
    generate_kwargs={
        "language": "urdu",      # Specify source language as Urdu
        "task": "translate",     # Change to "translate" for translation to English
        "temperature": 0.2,      # Lower = more deterministic output
    }
)

# Display the results
print(f"Translated Text: {result['text']}")

Device set to use cpu
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


Translated Text:  Assalam O Alikum, I am your representative artificial intelligence coach.


<img align=right src="../images/pl6.png" width="1000">

# 2. <span style='background :lightgreen' >Speech Generation: Text → Speech</span>

Text-to-Speech generation enables AI systems to convert written input into natural-sounding speech or even music. These models can produce lifelike voices, clone speaker styles, or generate musical compositions, opening up applications in accessibility, entertainment, and personalized content creation.
- **Closed Source:**
    - **OpenAI TTS:** OpenAI’s high-quality text-to-speech model producing natural, expressive voices in real time. (https://openai.com/index/tts/)
    - **ElevenLabs:** Industry-leading AI voice platform offering lifelike speech synthesis, voice cloning, and multilingual TTS. (https://elevenlabs.io/)
    - **Microsoft Azure Speech:** Cloud service for TTS with customizable voices, neural synthesis, and enterprise integration. (https://azure.microsoft.com/en-us/products/ai-services/text-to-speech)
- **Open Source:**
    - **Bark:** Suno’s transformer-based model for realistic TTS with non-verbal sounds, music, and multilingual capabilities. (https://github.com/suno-ai/bark)
    - **Tortoise TTS:** Open-source model focusing on ultra-realistic speech generation and voice cloning, though slower at inference. (https://github.com/neonbjb/tortoise-tts)
    - **XTTS-v2:** Coqui’s cross-lingual TTS model supporting multiple languages and speaker adaptation with open weights. (https://huggingface.co/coqui/XTTS-v2)

### Using OpenAI's `tts-1`
- **TTS-1** is OpenAI’s hosted text-to-speech model that converts written text into natural-sounding speech. It is not open-source and is available only through the OpenAI API.
- It can generate speech in multiple languages, including English, French, German, Spanish, Chinese, Japanese, Korean, Russian, and many more.
- The model supports expressive, human-like voices, including different speaking styles and tones depending on the selected voice preset.
- TTS-1 can be used for high-quality voiceovers, narration, interactive assistants, and real-time audio output (possible through streaming).
- The API is useful for:
    - Creating audiobooks, e-learning content, and multilingual educational material
    - Building voice interfaces for chatbots and applications
    - Generating narration for videos, podcasts, tutorials, and digital media content
    - Developing accessibility tools for individuals with low vision or reading difficulties
    - Enhancing games and immersive experiences (e.g., VR narration, character voices)
    - Helps with language learning, offering pronunciation guides or listening practice in a natural-sounding voice.
- OpenAI client provides two main endpoints for the TTS-1 model:
    - **`audio.speech.create()`:** Converts text → speech in the same language as the input text. Performs high-quality text-to-audio synthesis using lifelike voices. Example: English text → English speech, French text → French speech, Chinese text → Chinese speech. Note: Output language always matches the input text language (no automatic translation).
    - **`audio.speech.with_streaming_response.create()`:** Provides real-time streaming text → speech, allowing applications to start playing generated audio before the full synthesis is complete. Ideal for interactive assistants, live narration, or quick audio previews. Same behavior as above regarding language: the model reads the text as-is and does not translate.

In [6]:
# Using audio.speech.create()
from IPython.display import Audio 
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)   

# Generte audio of above text in English language using `client.audio.speech.create()` method which returns raw audio bytes
response = client.audio.speech.create(
                             model="tts-1",                                                               # Required: can call tts-1-hd for better quality
                             input="Hello students, welcome to learning Generative AI with Arif Butt.",   # Required: The text you want converted into speech. Can be string or array of text segments.    
                             voice="alloy"                                                                # Specifies which voice to use. Available voices: alloy, verse, coral, onxy, nova, shimmer, etc.
                            )

# Save the audio in a file named tts.mp3
with open('../data/audios/tts-generated-audio1.mp3', 'wb') as f:
    f.write(response.content)

# Play the audio in the notebook
Audio(response.content)

In [20]:
# Using audio.speech.create()
from IPython.display import Audio 
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)   

# Generte audio of above text in English language using `client.audio.speech.create()` method which returns raw audio bytes
response = client.audio.speech.create(
                             model="tts-1",                                                              
                             input="پیارے طلباء، میں آپ سب کو جنریٹو آرٹیفیشل انٹیلی جنس کے کورس میں خوش آمدید کہتا ہوں۔ میں آپ کا انسٹرکٹر عارف بٹ ہوں۔",   
                             voice="alloy"                                                             
                            )

# Save the audio in a file named tts.mp3
with open('../data/audios/tts-generated-audio2.mp3', 'wb') as f:
    f.write(response.content)

# Play the audio in the notebook
Audio('../data/audios/tts-generated-audio2.mp3') # loads and plays audio from a saved file on disk

### Using `audio.speech.with_streaming_response.create()`
- The TTS model starts generating audio immediately
- Audio is sent back in small byte chunks
- Your code receives each chunk as soon as it’s ready
- You don’t wait for the full audio before receiving data
- This is ideal for:
    - Real-time assistants
    - Low-latency voice apps
    - Live narration

In [10]:
# Using audio.speech.with_streaming_response.create()
from IPython.display import Audio
from openai import OpenAI

client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)

# Stream the audio from tts-1
with client.audio.speech.with_streaming_response.create(
                                                    model="tts-1",
                                                    input="Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit development, excelling in identifying and mitigating critical security risks across diverse platforms. His deep understanding of software architectures, memory corruption techniques, and attack vectors strengthens his ability to proactively enhance cybersecurity defences.A dedicated and results-driven professional, Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. His expertise in cybersecurity and emerging technologies enables him to contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.",
                                                    voice="alloy",
    
                                            ) as resp:
                                                    audio_bytes = b""               # collect bytes as they arrive
                                                    for chunk in resp.iter_bytes(): # iter_bytes() yields small audio chunks as soon as they are generated by the model (real-time streaming)
                                                        audio_bytes += chunk

# Play the streamed audio immediately (no saving)
Audio(audio_bytes, autoplay=True)     # plays in-memory audio bytes (ideal for streaming)

### Using Google Text-to-Speech (gTTS)
- **gTTS (Google Text-to-Speech)** is an open-source Python library that interfaces with Google Translate’s TTS API to convert written text into natural-sounding speech.
- It is open-source and free to use, but relies on Google’s hosted TTS service. It sends text to Google Translate’s TTS endpoint and returns audio
- It supports multiple languages, including English, French, German, Spanish, Chinese, Japanese, Korean, Russian, Hindi, Urdu, Arabic, and many more.
- gTTS can produce different accents and regional pronunciations by adjusting the tld (top-level domain) parameter, e.g., 'com' → American, 'co.uk' → British, 'co.in' → Indian.
- The library allows controlling speaking speed via the slow parameter (True for slow speech, False for normal).
- gTTS is suitable for:
    - Generating voiceovers for videos, tutorials, and presentations
    - Creating educational content and language learning material
    - Developing accessibility tools for people with low vision or reading difficulties
    - Narrating articles, e-books, or interactive applications
    - Producing multilingual audio content quickly and efficiently
- gTTS usage workflow:
    - gTTS() object creation: Prepares the text, selects language, accent, and speech rate. Does not generate audio immediately.
    - save(filename) method: Sends the text to Google TTS API and saves the resulting audio as an MP3 file.
    - Audio playback: Can be played in Python using libraries like IPython.display.Audio() or any MP3 player.
- **Notes:**
    - Unlike OpenAI’s tts-1, gTTS does not provide streaming playback; the full audio must be generated before it can be played.
    - gTTS does not offer multiple expressive voices or styles; the voice is determined by Google’s TTS engine and tld.

In [25]:
from gtts import gTTS              # gTTS: Library that uses Google Translate's TTS API Hosted at 'https://translate.google.co.in/' 
from IPython.display import Audio  # Audio: To play audio inside Jupyter Notebook

#  Input text you want to convert to speech
text = "Hello students, welcome to learning Generative AI with Arif Butt."

# The gTTS() function returns a gTTS object that does not generate speech immediately rather stores settings (customize accents, speed, preprocessing, cleaning and tokenization). 
# Actual audio is generated only when you call the save() method on this returned object, when it sends the object to Google TTS API and saves output audio
response = gTTS(
            text=text,        # Required: Text input string to convert to speech
            lang='en',        # Default = 'en') Has support of lot of languages like 'en': 'English', 'ur': 'Urdu',, 'ar': 'Arabic', 'de': 'German', 'hi': 'Hindi' and so on
            slow=False,       # Default = False means Normal speaking rate. Can be set to  True for slow speech
            tld='co.in',      # Default = 'com', that specifies US accent. Changing it gives regional accents. Top-level domain can have different values like 'com' → American,  'co.uk' → British, 'com.au' → Australian, 'co.in' → Indian
            lang_check=True,  # Default = True → Verify that 'lang' is supported
            )



# Save the audio in a file named tts.mp3
#response.save("../data/audios/gtts-generated-audio1.mp3")
with open('../data/audios/gtts-generated-audio1.mp3', 'wb') as f:
    response.write_to_fp(f)

# Play audio inside Jupyter Notebook
Audio(filename = "../data/audios/gtts-generated-audio1.mp3", autoplay=False)

In [26]:
from gtts import gTTS        
from IPython.display import Audio 

#  Input text you want to convert to speech
text = "پیارے طلباء، میں آپ سب کو جنریٹو آرٹیفیشل انٹیلی جنس کے کورس میں خوش آمدید کہتا ہوں۔ میں آپ کا انسٹرکٹر عارف بٹ ہوں۔"

# The gTTS() function returns a gTTS object that does not generate speech immediately rather stores settings (customize accents, speed, preprocessing, cleaning and tokenization). 
# Actual audio is generated only when you call the save() method on this returned object, when it sends the object to Google TTS API and saves output audio
response = gTTS(
            text=text,                 # Required: Text input string to convert to speech
            lang='ur',                 # Default = 'en') Has support of lot of languages like 'en': 'English', 'ur': 'Urdu',, 'ar': 'Arabic', 'de': 'German', 'hi': 'Hindi' and so on
            slow=False,                # Default = False means Normal speaking rate. Can be set to  True for slow speech
            tld='co.in',               # Default = 'com', that specifies US accent. Changing it gives regional accents. Top-level domain can have different values like 'com' → American,  'co.uk' → British, 'com.au' → Australian, 'co.in' → Indian
            lang_check=True,            # Default = True → Verify that 'lang' is supported
            )
# Save the generated speech to an MP3 file
response.save("../data/audios/gtts-generated-audio2.mp3")

# Play audio inside Jupyter Notebook
Audio(filename = "../data/audios/gtts-generated-audio2.mp3", autoplay=False)

### Using Open-Source Models wiht Transformers Pipeline
- You can create a TTS pipeline in Hugging Face using the `pipeline("text-to-speech", model_name)` method. Some popular TTS models are:
    - **`suno/bark-small`:** (https://huggingface.co/suno/bark-small) Lightweight TTS model from Suno.ai that generates natural-sounding English speech, optimized to run on CPU or low-VRAM machines.
    - **`facebook/mms-tts-eng`:** (https://huggingface.co/facebook/mms-tts-eng) Massively Multilingual Speech model from Facebook AI, supporting over 1100 languages, suitable for high-quality speech synthesis in many languages.
    - **`microsoft/speecht5_tts`:** (https://huggingface.co/microsoft/speecht5_tts) Transformer-based TTS model from Microsoft that produces expressive speech and can optionally use speaker embeddings for voice customization.
- The pipeline returns a waveform (numpy array) + sample rate. You can save it with soundfile and play it inline using Audio.

In [ ]:
from transformers import pipeline
from IPython.display import Audio

# Load Bark TTS pipeline
tts_pipeline = pipeline("text-to-speech", model="suno/bark-small", device=-1)

# Input text (can include special commands like [clears throat], [laughs])
text = "Hello students, welcome to learning Generative AI with Arif  Butt."

# Generate speech
speech = tts_pipeline(text)

# NOTE (transformers v5): the TTS pipeline now returns a FLAT 1-D waveform.
# In v4 it was shaped (1, N) so the code indexed [0]; doing that in v5 yields a single
# float32 sample and raises "Array audio input must be a 1D or 2D array".
# Play directly without saving the audio file on disk
Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
from transformers import pipeline
from scipy.io.wavfile import write
from IPython.display import Audio

# Load Bark TTS pipeline
tts_pipeline = pipeline("text-to-speech", model="suno/bark-small", device=-1)

# Input text (can include special commands like [clears throat], [laughs])
text = "Let me give you a funny statement, [clears throat], I shot an elephant wearning pajamas [laughs]."

# Generate speech
speech = tts_pipeline(text)

# Save audio file on disk
# NOTE (transformers v5): waveform is a flat 1-D array now - do not index it with [0]
write('../data/audios/speech-generatedby-bark-small.wav', 
      rate=speech["sampling_rate"], 
      data=speech["audio"])

# Load autio file from disk and play in notebook
Audio('../data/audios/speech-generatedby-bark-small.wav')

# 3. Sample Project (Cross-Modal Chaining)

- Input: Audio from the user via Gradio microphone component.
- Processing: Send audio → Groq-hosted LLM using the ask_groq function.
- Output: Receive text from Groq → convert to speech using gTTS.
- Playback: Play generated speech in Gradio.

```
Microphone
   ↓
Gradio Audio (filepath)
   ↓
Whisper-base (ASR)
   ↓
Groq-hosted LLM
   ↓
Text response
   ↓
gTTS
   ↓
Audio output
```

In [3]:
import os
from dotenv import load_dotenv
import gradio as gr
from transformers import pipeline
from gtts import gTTS
from openai import OpenAI


# Load environment variables
load_dotenv("../keys/.env", override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
# Groq client (OpenAI-compatible API)
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

# Whisper ASR (open-source, local)
transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=-1)

# Function to call Groq-hosted LLM
def ask_groq(
        user_prompt: str,
        developer_prompt: str = "You are a helpful assistant. Keep answers concise.",
        model: str = "llama-3.3-70b-versatile",
        max_output_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 1.0,
        ):
                response = client.responses.create(
                                                    model=model,
                                                    input=[{"role": "developer", "content": developer_prompt}, {"role": "user", "content": user_prompt}],
                                                    max_output_tokens=max_output_tokens,
                                                    temperature=temperature,
                                                    top_p=top_p,
                                                    )

                return response.output_text

# Main chatbot function (audio → text → LLM → audio)
def voice_chatbot(audio_path):
    if audio_path is None:
        return "No audio received.", None
    # 1. Speech → Text (Whisper)
    transcription = transcriber(audio_path)
    user_text = transcription["text"]
    # 2. Text → Groq LLM
    bot_text = ask_groq(user_text)
    # 3. Text → Speech (gTTS)
    output_audio_path = "bot_response.mp3"
    tts = gTTS(text=bot_text, lang="en")
    tts.save(output_audio_path)
    return bot_text, output_audio_path

# Gradio UI 
with gr.Blocks() as demo:
    gr.Markdown("""
    <h1 align=center>Vice Chatbot (Whisper + Groq + gTTS)</h1>
    """)
    with gr.Row():
        mic_input = gr.Audio(
            label="Speak",
            type="filepath"   # IMPORTANT for Whisper
        )

    bot_text_output = gr.Textbox(
        label="Bot Text Response",
        interactive=False
    )

    bot_audio_output = gr.Audio(
        label="Bot Audio Response",
        type="filepath",
        interactive=False
    )

    mic_input.change(
        fn=voice_chatbot,
        inputs=mic_input,
        outputs=[bot_text_output, bot_audio_output]
    )

demo.launch(inbrowser=True)

Device set to use cpu


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# 7. To Do:

## Video → Text (Video Understanding)
Video-to-text models extend visual understanding to dynamic sequences by analyzing both spatial and temporal information. They can interpret actions, events, and context across frames, enabling applications such as video summarization, content moderation, and accessibility.
- **Closed Source:**
    - **GPT-4o:** Multimodal model by OpenAI with strong capabilities to interpret video + audio + vision + text for temporal reasoning.
    - **Gemini Pro Vision:** Google DeepMind’s high-capacity vision-enabled version of Gemini for interpreting video content.
    - **Claude 3.5 Sonnet:** Anthropic’s multimodal model with enhanced video understanding and reasoning.
- **Open Source:**
    - **Video-LLaVA:** Unified visual-language model that aligns image & video inputs into the same representation space, allowing mixed image+video question answering.
    - **Chat-UniVi:** Model that uses unified visual tokens for both images and videos, enabling efficient temporal + spatial reasoning.
    - **Video-XL:** Extra-Long Vision-Language model for hour-scale video understanding, compressing visual input while preserving fine detail. 
- **Example Prompts:**
    - Upload cooking video → “Summarize the recipe and cooking steps.”
    - Upload sports clip → “Describe the key moments and player actions.”
- **Key Takeaway:** Models can analyze temporal visual sequences (i.e. video), track changes over time, understand context, and extract meaning from moving images in addition to static frames.

##  Text → Video Generation
Text-to-video models transform written descriptions into dynamic moving visuals. By combining advances in diffusion and multimodal learning, these systems can generate short video clips with realistic motion, artistic styles, or cinematic effects directly from prompts.
- **Closed Source:**
    - **OpenAI Sora:** OpenAI’s text-to-video model that generates short clips (up to ~20 seconds, 1080p) from text prompts, or remixes/extends video/image inputs. (https://openai.com/sora/)
    - **RunwayML Gen-2:** A commercial video generation tool for stylized text→video outputs (known for strong visuals and creative flexibility).
    - **Pika Labs:** Proprietary system for generating animated video content via text prompts, with a focus on accessible tools for creators.
- **Open Source:**
    - **ModelScope:** An open framework / repository of video generation models and tools for academic & community use.
    - **Zeroscope:** A powerful open-source text-to-video model (v2 and XL) that generates short realistic videos from prompts, without watermarks, in aspect ratios close to 16:9. (https://zeroscopeai.com/)
    - **AnimateDiff:** Open community tools / diffusion-based pipelines to animate text-to-image outputs or generate short video motion from text/image inputs.
- **Example Prompts:**
    - “A time-lapse of a flower blooming in a garden.”
    - “A cat walking through a cyberpunk alley at night.”
- **Key Takeaway:** From text descriptions alone, modern systems can generate moving visual narratives — dynamic scenes, changing lighting & motion — bridging static imagery and video storytelling.